In [1]:
import torch
import sys
import os
import tensorflow as tf
from ai_edge_quantizer import quantizer, recipe
import yaml

from ai_edge_litert.interpreter import Interpreter
from ai_edge_quantizer.utils import tfl_interpreter_utils
import litert_torch

sys.path.append(os.path.abspath(".."))
from models.network import BCMel, BCGabor
from data.yellowhammer import TrainingDataset

W0415 15:00:17.730000 3432 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/torch_xla2/distributed.py:106: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda` or `xpu`. Please specify it via the `devices` argument of `register_backend`.
  dist.Backend.register_backend("jax", ProcessGroupJax)
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ckpt_path = "/Users/zhang/MuReNN/yellowhammer_outputs/BCResnet/BCGabor/last.ckpt"
config_path = "/Users/zhang/MuReNN/yellowhammer_outputs/BCResnet/BCGabor/lightning_logs/version_0/hparams.yaml"

with open(config_path, 'r') as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

checkpoint = torch.load(ckpt_path, map_location='cpu')
print(checkpoint['state_dict'].keys())
network = BCGabor(**cfg['configuration_dict']['model_settings'])

network_weights = {k.replace("network.", ""): v for k, v in checkpoint['state_dict'].items()}
network.load_state_dict(network_weights)
network.eval()

odict_keys(['network.fb.gaborconv.kernel', 'network.net.cnn_head.0.weight', 'network.net.cnn_head.1.weight', 'network.net.cnn_head.1.bias', 'network.net.cnn_head.1.running_mean', 'network.net.cnn_head.1.running_var', 'network.net.cnn_head.1.num_batches_tracked', 'network.net.BCBlocks.0.0.f2.0.block.0.weight', 'network.net.BCBlocks.0.0.f2.0.block.1.weight', 'network.net.BCBlocks.0.0.f2.0.block.1.bias', 'network.net.BCBlocks.0.0.f2.0.block.1.running_mean', 'network.net.BCBlocks.0.0.f2.0.block.1.running_var', 'network.net.BCBlocks.0.0.f2.0.block.1.num_batches_tracked', 'network.net.BCBlocks.0.0.f2.1.block.0.weight', 'network.net.BCBlocks.0.0.f2.1.block.1.weight', 'network.net.BCBlocks.0.0.f2.1.block.1.bias', 'network.net.BCBlocks.0.0.f2.1.block.1.running_mean', 'network.net.BCBlocks.0.0.f2.1.block.1.running_var', 'network.net.BCBlocks.0.0.f2.1.block.1.num_batches_tracked', 'network.net.BCBlocks.0.0.f1.0.block.0.weight', 'network.net.BCBlocks.0.0.f1.0.block.1.weight', 'network.net.BCBlocks

BCGabor(
  (fb): Gabor(
    (gaborconv): GaborConv()
  )
  (net): BCResNets(
    (cnn_head): Sequential(
      (0): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (BCBlocks): ModuleList(
      (0): ModuleList(
        (0): BCResBlock(
          (f2): Sequential(
            (0): ConvBNReLU(
              (block): Sequential(
                (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (2): ReLU(inplace=True)
              )
            )
            (1): ConvBNReLU(
              (block): Sequential(
                (0): Conv2d(16, 16, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0), groups=16, bias=False)
                (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_run

In [5]:
dummy_input = (torch.randn(1, 30720),)
tflite_model = litert_torch.convert(network, dummy_input)
tflite_model.export("bc_gabor.tflite")

/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: Lazy import of LazyModule(package=None, target=speechbrain.integrations.k2_fsa, loaded=False) failed
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/litert_torch/_convert/signature.py:52: FutureWarning: `treespec.children_specs` is deprecated. Use `treespec.child(index)` to access a single child, or `treespec.children()` to get all children.
  args_spec, kwargs_spec = spec.children_specs
/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/litert_torch/_convert/signature.py:58: FutureWarning: `treespec.children_specs` is deprecated. Use `treespec.child(index)` to access a single child, or `treespec.children()` to get all children.
  kwargs_spec.children_specs, kwargs_spec.context
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: Lazy import of LazyModule(package=None, target=speechbrain.integrations.k2_fsa, loaded=False) failed
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: Lazy import of LazyModule(package=None, target=speechbrain.integrations.k2_fsa, loaded=False) failed
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: Lazy import of LazyModule(package=None, target=speechbrain.integrations.k2_fsa, loaded=False) failed
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: Lazy import of LazyModule(package=None, target=speechbrain.integrations.k2_fsa, loaded=False) failed
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: Lazy import of LazyModule(package=None, target=speechbrain.integrations.k2_fsa, loaded=False) failed
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
INFO:tensorflow:Assets written to: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpvrthqjho/assets


INFO:tensorflow:Assets written to: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpvrthqjho/assets
W0000 00:00:1776258065.972490 33537314 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776258065.972502 33537314 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776258065.972658 33537314 reader.cc:83] Reading SavedModel from: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpvrthqjho
I0000 00:00:1776258065.974645 33537314 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1776258065.974649 33537314 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpvrthqjho
I0000 00:00:1776258065.995378 33537314 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1776258066.114196 33537314 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpvrthqjho
I0000 00:00:1776258066.148096 33537314 loader.

In [7]:
interpreter = Interpreter(model_path="bc_gabor.tflite")
input_details = interpreter.get_input_details()
import re
def clean_input_layer_name(name: str) -> str:
    name = name.split(":")[0]
    name = re.sub(r"^serving_default_", "", name)
    return name

data_dir = '/Users/zhang/MuReNN/YH_data_with_aug/train'
dataset = TrainingDataset(data_dir)
# Randomly select 1 samples for calibration
idx = torch.randperm(len(dataset))[:1]
dataset = [dataset[i] for i in idx]
calibration_samples = []

for i, sample in enumerate(dataset):
    calibration_samples.append({
        clean_input_layer_name(input_details[0]['name']): sample['waveform'].reshape(1, 1, -1),
    })  

calibration_data = {
    tfl_interpreter_utils.DEFAULT_SIGNATURE_KEY: calibration_samples,
}

In [8]:
qt_static = quantizer.Quantizer("bc_gabor.tflite")
qt_static.load_quantization_recipe(recipe.static_wi8_ai16())
calibration_result = qt_static.calibrate(calibration_data)
qt_static.quantize(calibration_result).export_model("bc_gaborINT16.tflite", overwrite=True)

/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/ai_edge_litert/interpreter.py:465: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
